# AgentIR: experiments for the Matan meeting

One basic experiment per improvement idea from the PlugIR / A2R inspection deck, so we bring numbers instead of suggestions.

| Idea | Experiment | Needs | Time on a T4 |
|---|---|---|---|
| 02 Keep what the user said | **E2a** rewrite drift in PlugIR's released dialogs | text only | 1 min |
| 01 Show the right images | **E1** evidence each method shows the questioner | BLIP index | ~15 min (500 dialogs) |
| 02 Keep what the user said | **E2b** rank cost of rewrite errors + text-only fixes | BLIP index | ~5 min |
| 01 Show the right images | **E1b** PlugIR vs rank-aware representatives vs fixed pool, end to end | Qwen | hours, start small |
| 02 Keep what the user said | **E2c** same Q&A, different rewriters | Qwen, reuses E1b's PlugIR run | ~20 min |
| 03 Questions beyond the 10 | **E3** does IG on the 10 shown images transfer to other top-100 images? | Qwen VLM | ~45 min (20 dialogs) |

Do the no-LLM parts (sections 1–4) first; they already answer most of what Matan asked. **Runtime → Change runtime type → T4 GPU.** See `experiments/README.md` for what each number means.

## 0. Setup

In [ ]:
BRANCH = 'experiments-matan'

%cd /content
!git clone -b {BRANCH} https://github.com/nopharshalom/ChatIR-Cornell2026 2>/dev/null || (cd ChatIR-Cornell2026 && git fetch && git checkout {BRANCH} && git pull)
%cd /content/ChatIR-Cornell2026
!pip install -q -e ".[dev]" spacy && python -m spacy download -q en_core_web_sm
!pytest -q | tail -1

In [ ]:
USE_DRIVE = False

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/chatir-cornell2026'
    !mkdir -p "{SAVE_DIR}/cache" "{SAVE_DIR}/runs"
    !rm -rf cache runs && ln -s "{SAVE_DIR}/cache" cache && ln -s "{SAVE_DIR}/runs" runs
else:
    SAVE_DIR = None
    !mkdir -p cache runs
!ls -la cache runs

## 1. Rewrite drift, text only (E2a)
No GPU needed. Runs on PlugIR's 2,064 released rewritten queries.

In [ ]:
PLUGIR = 'dialogues/ours_final_q_n_5_recall_hitting_10_thres_low_500_recon_true_referring_true_filtering_true_select_true_reconed.json'
!python experiments/e2_rewrite_drift_text.py {PLUGIR} --out runs/e2a

## 2. Images and retrieval index
Same as `AgentIR_Colab.ipynb` sections 3–5. If you already have `cache/index_blip.pt`, upload it and skip the image download only if you will not run E1b/E3 (those need the images).

In [ ]:
if not os.path.isdir('/content/data/unlabeled2017'):
    !mkdir -p /content/data
    !wget -q --show-progress -O /content/data/unlabeled2017.zip http://images.cocodataset.org/zips/unlabeled2017.zip
    !unzip -q /content/data/unlabeled2017.zip -d /content/data && rm /content/data/unlabeled2017.zip

# Checks all 50,000 protocol images exist; fills any VisDial targets missing from COCO.
!agentir prepare-data --data-dir /content/data

In [ ]:
B = '-c configs/base.yaml -c configs/backends/colab.yaml'

In [ ]:
# Reuse PlugIR_Colab.ipynb's image embeddings if present (upload cache/corpus_blip.pth), else embed from scratch.
if os.path.exists('cache/index_blip.pt'):
    print('index already built')
elif os.path.exists('cache/corpus_blip.pth'):
    !agentir build-index {B} --from-plugir-cache cache/corpus_blip.pth
else:
    !agentir build-index {B}

**Sanity check: expect BRI ≈ 0.767** for PlugIR's released dialogs.

In [ ]:
!agentir eval-dialogs {B} {PLUGIR} --split --name check-plugir-released

## 3. Is the questioner shown the right images? (E1, no LLM)
`--limit 500` keeps it ~15 min; drop it for all 2,064 dialogs.

In [ ]:
!python experiments/e1_candidate_evidence.py {B} --dialogs {PLUGIR} --limit 500 --out runs/e1

## 4. Do rewrite errors cost rank, and do text-only fixes help? (E2b, no LLM)

In [ ]:
!python experiments/e2_rewrite_retrieval.py {B} --dialogs {PLUGIR} --drift runs/e2a/e2a_metrics.json --out runs/e2b

## 5. Start Qwen3.5-9B (Ollama)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -2 /tmp/ollama_install.log

In [ ]:
import subprocess, time, urllib.request

env = dict(os.environ,
           OLLAMA_NUM_PARALLEL='4',        # serve 4 sessions at once (configs/backends/colab.yaml uses 4 workers)
           OLLAMA_CONTEXT_LENGTH='8192',   # PlugIR prompts + one image fit comfortably
           OLLAMA_KEEP_ALIVE='-1')         # keep the model loaded between calls
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env,
                               stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2); print('ollama is up'); break
    except Exception:
        time.sleep(1)
else:
    print('ollama did not start; see /content/ollama.log')

In [ ]:
!ollama pull qwen3.5:9b
# Latency, no reasoning leaking into answers, and image input for the simulated user:
!agentir check-llm {B} --image /content/data/unlabeled2017/000000185565.jpg
!ollama ps    # PROCESSOR should say 100% GPU

## 6. End-to-end check of idea 01 (E1b)

Start with **20 dialogs × 10 rounds** (`N = 20`). The PlugIR run is also the input for E2c. Each run resumes if interrupted and LLM calls are cached, so a later N=50 run reuses this work. Run E3 (section 8) first if time is short; it is cheaper.

In [ ]:
N = 20
for exp in ['plugir', 'e1_rep_rank', 'e1_fixed_pool']:
    !agentir run {B} -c configs/experiments/{exp}.yaml --set data.limit={N}

## 7. Same questions and answers, different rewriters (E2c)

In [ ]:
import glob, os
plugir_run = max(glob.glob('runs/plugir-*'), key=os.path.getmtime)
print(plugir_run)
!python experiments/e2_replay_reformulators.py {B} --run {plugir_run} --out runs/e2c

## 8. Do questions chosen on 10 images help on the wider pool? (E3)

In [ ]:
!python experiments/e3_question_transfer.py {B} --dialogs {PLUGIR} --limit 20 --out runs/e3

## 9. One summary for the meeting

In [ ]:
import json, glob
def show(path, key='table'):
    for p in glob.glob(path):
        m = json.load(open(p)); print(f'== {p}'); print(m.get(key, ''), '\n')
show('runs/e1/e1_metrics.json'); print(json.dumps(json.load(open('runs/e1/e1_metrics.json'))['coverage'], indent=1))
show('runs/e2b/e2b_metrics.json'); print(json.dumps(json.load(open('runs/e2b/e2b_metrics.json'))['regressions'], indent=1))
show('runs/e2c/e2c_metrics.json')
show('runs/e3/e3_metrics.json'); print(json.dumps(json.load(open('runs/e3/e3_metrics.json'))['agreement'], indent=1))
for d in sorted(glob.glob('runs/*/metrics.json')):
    m = json.load(open(d))
    if 'name' in m: r = m['num_rounds']; print(f"{m['name']:<16} n={m['num_sessions']:<4} BRI={m['bri']:.3f} Hits@10={m['hits']['10'][r]:.1f} R@10={m['recall']['10'][r]:.1f}")

In [ ]:
!zip -qr /content/experiments_results.zip runs cache/index_blip.pt /content/llm_cache.sqlite 2>/dev/null; ls -lh /content/experiments_results.zip
from google.colab import files
files.download('/content/experiments_results.zip')